# RF Final Predictions - No 'Time Travel'


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

from kaggle_games.pipelines import PredictiveSteamEngineer
from kaggle_games.split_data import get_split_data

In [ ]:
data_path = Path.cwd().parent / "data" / "games.json"
df = pd.read_json(data_path, orient="index")
df.replace(r"^\s*$", np.nan, regex=True, inplace=True)

X_train, X_test, y_train, y_test = get_split_data(df)

In [ ]:
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


prep_steps = [
    ("engineer", PredictiveSteamEngineer()),
    ("imputer", SimpleImputer(strategy="median")),
]

pipe_baseline_rf = Pipeline(
    steps=prep_steps
    + [("rf", RandomForestClassifier(random_state=42, n_jobs=-1))]
)
pipe_cw_rf = Pipeline(
    steps=prep_steps
    + [
        (
            "rf",
            RandomForestClassifier(
                class_weight="balanced", random_state=42, n_jobs=-1
            ),
        )
    ]
)
pipe_smote_rf = ImbPipeline(
    steps=prep_steps
    + [
        ("smote", SMOTE(random_state=42)),
        ("rf", RandomForestClassifier(random_state=42, n_jobs=-1)),
    ]
)

param_grid_rf = {
    "rf__n_estimators": [100, 200],
    "rf__max_depth": [10, 20, None],
}


def run_search_rf(pipeline, name):
    print(f"--- Tuning {name} ---")
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid_rf,
        n_iter=3,
        cv=cv_strategy,
        scoring="roc_auc",
        random_state=42,
        n_jobs=None,
        verbose=1,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_


best_baseline_rf = run_search_rf(pipe_baseline_rf, "RF Baseline (Predictive)")
best_cw_rf = run_search_rf(pipe_cw_rf, "RF Class Weights (Predictive)")
best_smote_rf = run_search_rf(pipe_smote_rf, "RF SMOTE (Predictive)")

models_rf = {
    "Baseline": best_baseline_rf,
    "Class Weights": best_cw_rf,
    "SMOTE": best_smote_rf,
}


def evaluate_model(model, X_test, y_test, title):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"\n=== {title} ===")
    print(classification_report(y_test, y_pred, zero_division=0))

    return {
        "y_pred": y_pred,
        "y_proba": y_proba,
        "auc": roc_auc_score(y_test, y_proba),
        "f1": f1_score(y_test, y_pred),
        "prec": precision_score(y_test, y_pred, zero_division=0),
        "rec": recall_score(y_test, y_pred, zero_division=0),
    }


eval_results_rf = {}
for name, model in models_rf.items():
    eval_results_rf[name] = evaluate_model(model, X_test, y_test, f"RF {name}")

plt.figure(figsize=(10, 7))
colors = {"Baseline": "gray", "Class Weights": "blue", "SMOTE": "red"}
for name, res in eval_results_rf.items():
    precision, recall, _ = precision_recall_curve(y_test, res["y_proba"])
    plt.plot(
        recall,
        precision,
        label=f"{name} (F1 = {res['f1']:.3f})",
        color=colors[name],
        linewidth=2,
    )

plt.title(
    "RF Precision-Recall Curve (Strictly Pre-Release)",
    fontweight="bold",
    fontsize=14,
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

rf_model = best_cw_rf.named_steps["rf"]
engineer = best_cw_rf.named_steps["engineer"]

feature_names = engineer.transform(X_train.head(1)).columns
importances = rf_model.feature_importances_

importance_df = pd.DataFrame(
    {"Feature": feature_names, "Importance": importances}
).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=importance_df.head(15), x="Importance", y="Feature", palette="viridis"
)
plt.title(
    "Top 15 Feature Importances (Random Forest + Class Weights)",
    fontweight="bold",
    fontsize=14,
)
plt.xlabel("Mean Decrease in Impurity (Gini Importance)", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.show()